<a href="https://colab.research.google.com/github/siddheshmm/Courses-Data/blob/main/IBM%20DL0320EN%20Applied%20Deep%20Learning%20Capstone%20Project/DL0320EN_4_1_CompareModels_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://cocl.us/DL0320EN_TOP_IMAGE">
    <img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0320EN/Assets/Images/Top.png" width="750" alt="IBM 10TB Storage">
</a>


<h1>Classifying European Money Denominations: Comparing Two Models</h1>


<h2>Table of Contents</h2>


<p>In this lab you will compare the <code>ResNet18</code> and <code>Densenet121</code></p>
<ul>
    <li><a href="#dataset">Create Dataset Class and Object</a></li>
    <li><a href="#pre">Load Pre-trained Model</a></li>
    <li><a href="#analyze">Analyze Models</a></li>
</ul>

<p>Estimated Time Needed: <b>25 min</b></p>
<hr>


<h2>Preparation</h2>


<a href="https://cocl.us/DL0320EN_storage">
    <img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0320EN/Assets/Images/ObjectStorage.png" width="750" alt="cognitive class">
</a>


Download the datasets you needed for this lab.


In [1]:
# You can comment out this box when you already have the dataset
# Step 1: Ctrl + A : Select all
# Step 2: Ctrl + / : Comment out all; if everything selected has been comment out alreaday, then uncomment all

# Download test dataset
!wget --quiet -O ./data/test_data_pytorch.tar.gz https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0320EN/Datasets/PyTorch/test_data_pytorch.tar.gz
!tar -xzf ./data/test_data_pytorch.tar.gz -C ./data --exclude '.*'

Import the PyTorch Modules needed in the lab.


In [8]:
# Import PyTorch Modules that will be used in the lab

import torch
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import pandas
from torchvision import transforms
import torch.nn as nn
torch.manual_seed(0)

Import Non-PyTorch Modules


In [9]:
# Import Non-PyTorch Modules that will be used in the lab

import time
from imageio import imread
from matplotlib.pyplot import imshow
import matplotlib.pylab as plt
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import random
import numpy as np

<hr>


<h2 id="dataset">Create Dataset Class and Object</h2>


In this section, you use the dataset class from the last section.


The denomination, file name and the class variable for the testing data are stored in the following CSV file.


In [10]:
# Url that contains CSV files and image dataset folder

test_csv_file = 'https://cocl.us/DL0320EN_TEST_CSV'
test_data_dir = './data/test_data_pytorch/'

Use the dataset class you created in the last lab.


In [11]:
# Create Dataset Class

class Dataset(Dataset):

    # Constructor
    def __init__(self, csv_file, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.data_name = pd.read_csv(csv_file)
        self.len = self.data_name.shape[0]

    # Get Length
    def __len__(self):
        return self.len

    # Getter
    def __getitem__(self, idx):
        img_name = self.data_dir + self.data_name.iloc[idx, 2]
        image = Image.open(img_name)
        y = self.data_name.iloc[idx, 3]
        if self.transform:
            image = self.transform(image)
        return image, y

<h3>Try</h3>


Use the constructor <code>compose</code> to perform the following sequence of transformations in the order they are given, call the object <code>composed</code>


In [12]:
# Construct the composed object for transforming the image
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
trans_step = [transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(mean, std)]


# Type your code here
composed = transforms.Compose(trans_step)



Double-click <b>here</b> for the solution.
<!--
composed = transforms.Compose(trans_step)
-->


Create a test dataset object using the CSV file stored in the variables the <code>test_csv_file</code>. The directories are stored in the variable <code>test_data_dir</code>. Set the parameter <code>transform</code> to the object <code>composed</code>.


In [13]:
# Create a test_dataset

test_dataset = Dataset(transform=composed
                       , csv_file=test_csv_file
                       , data_dir=test_data_dir)

<hr>


<h2 id="pre">Load Pre-trained Model</h2>


Load the <code>ResNet18</code> and <code>Densenet121</code> model you created from the last section


In [26]:
# Load pre-trained model
model = torch.load("resnet18_pytorch_model.pt")
model_des = torch.load("densenet121_pytorch_model.pt")

Print the structures of two models. You need to answer the questions in quiz based on the output here.


In [27]:
# Print model structure

print("ResNet18:\n", model)
print("Densenet121:\n", model_des)

Streaming output truncated to the last 5000 lines.
          [-9.8929e-03, -6.3410e-03, -6.5064e-03],
          [-3.4284e-03, -8.8969e-04, -8.9345e-03]],

         [[-1.9041e-02, -1.9522e-02, -2.1111e-02],
          [-1.9628e-02, -2.0088e-02, -1.7635e-02],
          [-1.8567e-02, -1.4694e-02, -1.3183e-02]]],


        [[[-1.5675e-02, -4.7860e-03, -8.8170e-03],
          [-1.1698e-02,  6.6762e-04, -5.5051e-03],
          [-1.5408e-02,  3.4478e-03, -2.1439e-03]],

         [[-1.4574e-02, -1.1842e-02, -1.3500e-02],
          [-2.1318e-02, -1.4492e-02, -1.3540e-02],
          [-2.4376e-02, -1.5521e-02, -1.5680e-02]],

         [[-1.3197e-02, -9.6210e-03, -8.4839e-03],
          [-1.5601e-02, -1.1779e-02, -9.2510e-03],
          [-1.2079e-02, -1.0084e-02,  2.2599e-03]],

         ...,

         [[-1.0114e-02,  3.6591e-03,  7.3630e-03],
          [-4.1775e-03, -4.5251e-03,  4.1620e-03],
          [-6.9823e-03, -4.6105e-03, -1.3163e-03]],

         [[-7.2246e-03, -1.0856e-02, -1.3128e-02],
  

Create a data loader object for the test data


In [28]:
# Set Data Loader object

test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=10)

<hr>


<h2 id="analyze">Analyze Models</h2>


<h3>Try</h3>


Find the error on the test data using the <code>ResNet18</code> model.


In [33]:
# Predict the data using ResNet18 model and print out accuracy

# Type your code here

correct = 0
accuracy = 0
N = len(test_dataset)
for x_test, y_test in test_loader:
    model.eval()
    z = model(x_test)
    _, yhat = torch.max(z.data, 1)
    correct += (yhat == y_test).sum().item()
accuracy = correct / N
print("Accuracy using ResNet18: ", accuracy)

Accuracy using ResNet18:  0.0


Double-click <b>here</b> for the solution.
<!--
correct = 0
accuracy = 0
N = len(test_dataset)
for x_test, y_test in test_loader:
    model.eval()
    z = model(x_test)
    _, yhat = torch.max(z.data, 1)
    correct += (yhat == y_test).sum().item()
accuracy = correct / N
print("Accuracy using ResNet18: ", accuracy)
-->


<h3>Try</h3>


Find the error on the test data using the <code>Densenet121</code> model


In [35]:
# Predict the data using densene model and print out accuracy

# Type your code here

correct = 0
accuracy = 0
N = len(test_dataset)
for x_test, y_test in test_loader:
    model_des.eval()
    z = model_des(x_test)
    _, yhat = torch.max(z.data, 1)
    correct += (yhat == y_test).sum().item()
accuracy = correct / N
print("Accuracy using Densenet121: ", accuracy)

AttributeError: 'collections.OrderedDict' object has no attribute 'eval'

Double-click <b>here</b> for the solution.
<!--
correct = 0
accuracy = 0
N = len(test_dataset)
for x_test, y_test in test_loader:
    model_des.eval()
    z = model_des(x_test)
    _, yhat = torch.max(z.data, 1)
    correct += (yhat == y_test).sum().item()
accuracy = correct / N
print("Accuracy using Densenet121: ", accuracy)
-->


<h3>Try</h3>


What model performed better on the test data


In [ ]:
# Write your answer here

I think the Densenet121 model performed better on test data

<a href="https://cocl.us/DLO0320EN_notebook_bott">
    <img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0320EN/Assets/Images/Bottom.png" width="750" alt="cognitive class">
</a>


<h2>About the Authors:</h2>

<a href="https://www.linkedin.com/in/joseph-s-50398b136/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDL0320ENSkillsNetwork929-2023-01-01">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.


Other contributors: <a href="https://www.linkedin.com/in/michelleccarey/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDL0320ENSkillsNetwork929-2023-01-01">Michelle Carey</a>, <a href="www.linkedin.com/in/jiahui-mavis-zhou-a4537814a">Mavis Zhou</a>, <a href="https://www.linkedin.com/in/yi-leng-yao-84451275/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDL0320ENSkillsNetwork929-2023-01-01">Yi Leng Yao</a>


<hr>


Copyright &copy; 2018 <a href="cognitiveclass.ai?utm_source=bducopyrightlink&utm_medium=dswb&utm_campaign=bdu">cognitiveclass.ai</a>. This notebook and its source code are released under the terms of the <a href="https://bigdatauniversity.com/mit-license/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDL0320ENSkillsNetwork929-2023-01-01">MIT License</a>.
